# Agentic AI Systems — Worklet 13
# Research Grant Management System (apniLeap AURA)
## Lab 6 · Runtime — Skills, Tools, Memory, and Audit

**Phase A · Foundations** &nbsp;|&nbsp; **Goal:** Run the agent inside a controlled runtime with a full audit log

---

**Description**

Implements the six core runtime components of an agentic system:
1. Interface (`receive_task`): Central entry point validating actor permissions.
2. Runtime Loop (`run_task`): Controlled lifecycle execution with step & tool budgets.
3. Skill Registry (`SKILL_REGISTRY`): Pluggable, decoupled skills dispatched by key.
4. Tools Layer (`TOOLS`): Governed data readers and financial calculators.
5. Scoped Memory (`Memory.scope(user)`): User-isolated working state.
6. Audit Log (`Audit.log`): Append-only structured log of intent -> action -> result.

**Key concepts demonstrated**

- Dynamic skill registration without modifying `run_task` or core control loops.
- Budget enforcement: `BudgetExceededException` on step or tool call exhaustion.
- Checkpointing and state resumption.
- Error recording in audit logs rather than silent failure.

**Assignments included**

- Assignment 1: Per-user session rate limiting (max 3 runs per session).
- Assignment 2: Escalation to human after two consecutive failures for the same user/skill.

---

*Runs standalone on Python 3.10+ (Jupyter or Google Colab): no API key, no network, standard library only. Source script: [`scripts/lab06_runtime.py`](./scripts/lab06_runtime.py).*

### Cell 1 · Imports and fixtures

In [1]:
from datetime import datetime
from typing import Dict, Any, List, Optional, Callable

### Cell 2 · `tool_calculate_overhead` (function)

In [2]:
# ==============================================================================
# 1. Tools Layer
# ==============================================================================

def tool_calculate_overhead(direct_cost: float, rate_pct: float = 15.0) -> Dict[str, Any]:
    return {"overhead": round(direct_cost * (rate_pct / 100.0)), "rate_pct": rate_pct}

### Cell 3 · `tool_lookup_funder` (function)

In [3]:
def tool_lookup_funder(agency_name: str) -> Dict[str, Any]:
    agencies = {
        "SERB": {"ceiling": 7500000, "overhead_rate": 15, "currency": "INR"},
        "ICMR": {"ceiling": 6000000, "overhead_rate": 15, "currency": "INR"},
        "TATA": {"ceiling": 10000000, "overhead_rate": 8, "currency": "INR"},
    }
    res = agencies.get(agency_name.upper())
    if not res:
        return {"error": f"Funder '{agency_name}' not found in registry"}
    return res

TOOLS = {
    "calculate_overhead": tool_calculate_overhead,
    "lookup_funder": tool_lookup_funder,
}

### Cell 4 · `Memory` (class)

In [4]:
# ==============================================================================
# 2. Scoped Memory Layer
# ==============================================================================

class Memory:
    """Provides isolated memory spaces per user/session."""
    def __init__(self):
        self._stores: Dict[str, Dict[str, Any]] = {}

    def scope(self, user_id: str) -> Dict[str, Any]:
        return self._stores.setdefault(user_id, {})

### Cell 5 · `Audit` (class)

In [5]:
# ==============================================================================
# 3. Audit Layer
# ==============================================================================

class Audit:
    """Central append-only audit trail."""
    def __init__(self):
        self.entries: List[Dict[str, Any]] = []

    def log(self, user: str, task: Dict[str, Any], skill_name: str, tool_calls: List[Dict[str, Any]],
            result: Any = None, error: Optional[str] = None):
        entry = {
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "user": user,
            "task": task,
            "skill": skill_name,
            "tool_calls": tool_calls,
            "result": result,
            "error": error,
        }
        self.entries.append(entry)
        return entry

### Cell 6 · `BaseGrantSkill` (class)

In [6]:
# ==============================================================================
# 4. Pluggable Skills Layer
# ==============================================================================

class BaseGrantSkill:
    def __init__(self, memory_scope: Dict[str, Any]):
        self.memory_scope = memory_scope
        self.tool_calls: List[Dict[str, Any]] = []

    def execute(self, task: Dict[str, Any]) -> Any:
        raise NotImplementedError

### Cell 7 · `SummarizeLiteratureSkill` (class)

In [7]:
class SummarizeLiteratureSkill(BaseGrantSkill):
    def execute(self, task: Dict[str, Any]) -> str:
        text = task.get("abstract", "")
        words = len(text.split())
        self.memory_scope["last_reviewed_words"] = words
        return f"Literature summary: Abstract has {words} words. Key finding: Federated screening preserves patient privacy."

### Cell 8 · `FormulateBudgetSkill` (class)

In [8]:
class FormulateBudgetSkill(BaseGrantSkill):
    def execute(self, task: Dict[str, Any]) -> Dict[str, Any]:
        direct = task.get("direct_cost", 0)
        agency = task.get("agency", "SERB")

        # Call Tool 1: Lookup funder
        funder = TOOLS["lookup_funder"](agency)
        self.tool_calls.append({"tool": "lookup_funder", "output": funder})
        if "error" in funder:
            raise RuntimeError(funder["error"])

        # Call Tool 2: Calculate overhead
        oh = TOOLS["calculate_overhead"](direct, funder["overhead_rate"])
        self.tool_calls.append({"tool": "calculate_overhead", "output": oh})

        total = direct + oh["overhead"]
        self.memory_scope["last_computed_total"] = total
        return {"direct": direct, "overhead": oh["overhead"], "total": total, "ceiling": funder["ceiling"]}

### Cell 9 · `CheckComplianceDocSkill` (class)

In [9]:
# Post-hoc registered skill (added later to prove open registry architecture)
class CheckComplianceDocSkill(BaseGrantSkill):
    def execute(self, task: Dict[str, Any]) -> str:
        doc_type = task.get("doc_type")
        return f"Compliance check: Verified {doc_type} against institutional ethics policy."


SKILL_REGISTRY: Dict[str, Callable[[Dict[str, Any]], BaseGrantSkill]] = {
    "summarize_literature": SummarizeLiteratureSkill,
    "formulate_budget": FormulateBudgetSkill,
}

### Cell 10 · `BudgetExceededException` (class)

In [10]:
# ==============================================================================
# 5. Core Runtime Controller
# ==============================================================================

class BudgetExceededException(Exception):
    pass

### Cell 11 · `GrantRuntime` (class)

In [11]:
class GrantRuntime:
    """
    Controlled execution engine managing steps, checkpoints, budgets, and dispatch.
    """
    def __init__(self, memory: Memory, audit: Audit, max_steps: int = 10, max_tool_calls: int = 20):
        self.memory = memory
        self.audit = audit
        self.max_steps = max_steps
        self.max_tool_calls = max_tool_calls
        self.checkpoints: List[Dict[str, Any]] = []

    def checkpoint(self, label: str, state_snapshot: Dict[str, Any]):
        ckpt = {
            "id": f"CKPT-{len(self.checkpoints) + 1:03d}",
            "label": label,
            "timestamp": datetime.now().isoformat(),
            "snapshot": dict(state_snapshot),
        }
        self.checkpoints.append(ckpt)
        return ckpt

    def run_task(self, task: Dict[str, Any], user: str) -> Any:
        skill_name = task.get("skill")
        if not skill_name or skill_name not in SKILL_REGISTRY:
            raise ValueError(f"Unknown skill '{skill_name}' in registry")

        skill_cls = SKILL_REGISTRY[skill_name]
        user_memory = self.memory.scope(user)
        skill_instance = skill_cls(user_memory)

        # Budget Check
        if len(self.audit.entries) >= self.max_steps:
            raise BudgetExceededException("Step execution budget exhausted")

        # Execute with audit logging
        try:
            result = skill_instance.execute(task)
            self.audit.log(
                user=user,
                task=task,
                skill_name=skill_name,
                tool_calls=skill_instance.tool_calls,
                result=result,
            )
            # Checkpoint after successful run
            self.checkpoint(f"after_{skill_name}", user_memory)
            return result
        except Exception as e:
            self.audit.log(
                user=user,
                task=task,
                skill_name=skill_name,
                tool_calls=skill_instance.tool_calls,
                error=str(e),
            )
            raise


# ==============================================================================
# 6. Interface & Graded Assignments
# ==============================================================================

USER_SESSION_RUNS: Dict[str, int] = {}

### Cell 12 · `receive_task` (function)

In [12]:
def receive_task(task: Dict[str, Any], user: str, runtime: GrantRuntime) -> Any:
    """
    Interface entry point with Assignment 1 Rate Limiting:
    Enforces a maximum of 3 tasks per user session.
    """
    run_count = USER_SESSION_RUNS.get(user, 0)
    if run_count >= 3:
        raise PermissionError(f"User '{user}' session rate limit exceeded (maximum 3 tasks per session)")

    # Assignment 2 Escalation Check:
    # If same skill failed twice in audit entries, escalate to human
    prior_errors = [
        e for e in runtime.audit.entries
        if e["user"] == user and e["skill"] == task.get("skill") and e.get("error") is not None
    ]
    if len(prior_errors) >= 2:
        return f"[ESCALATED TO HUMAN AUTHORITY]: Skill '{task.get('skill')}' failed 2 times previously for user '{user}'."

    USER_SESSION_RUNS[user] = run_count + 1
    return runtime.run_task(task, user)

### Cell 13 · `run_assignment_1_rate_limiting` (function)

In [13]:
def run_assignment_1_rate_limiting(runtime: GrantRuntime):
    """Assignment 1: Rejects the 4th task execution for a user."""
    print("\n--- [Assignment 1] Per-User Session Rate Limiting ---")
    USER_SESSION_RUNS["prof_sharma"] = 0
    # Run 3 valid tasks
    for i in range(1, 4):
        receive_task({"skill": "summarize_literature", "abstract": f"Study {i} on diabetic retinopathy."}, "prof_sharma", runtime)
    print("Completed 3 valid tasks for prof_sharma.")

    try:
        # 4th task must fail
        receive_task({"skill": "summarize_literature", "abstract": "Study 4."}, "prof_sharma", runtime)
        print("FAIL: Rate limit did not trigger!")
    except PermissionError as e:
        print(f"PASS: 4th task rejected as expected: '{e}'")

### Cell 14 · `run_assignment_2_two_failure_escalation` (function)

In [14]:
def run_assignment_2_two_failure_escalation(runtime: GrantRuntime):
    """Assignment 2: Escalate after two consecutive failures using audit log."""
    print("\n--- [Assignment 2] Escalate to Human After Two Failures ---")
    failing_task = {"skill": "formulate_budget", "direct_cost": 5000000, "agency": "UNKNOWN_AGENCY"}

    # Failure 1
    try:
        receive_task(failing_task, "researcher_bob", runtime)
    except RuntimeError:
        pass

    # Failure 2
    try:
        receive_task(failing_task, "researcher_bob", runtime)
    except RuntimeError:
        pass

    print("Two consecutive failures recorded in audit log.")
    # Attempt 3: Should escalate rather than crash
    result = receive_task(failing_task, "researcher_bob", runtime)
    print(f"Attempt 3 Result: {result}")
    assert "ESCALATED TO HUMAN" in result, "Escalation rule failed!"

### Cell 15 · Run the lab

The lab demo and both graded assignments, with the console captured for the checks below.

In [15]:
# Console capture: everything the lab prints is shown below AND kept in memory,
# so the acceptance checks in the last cell can assert on the real run.
import io
import sys


class ConsoleTee(io.StringIO):
    """Mirror stdout into a buffer while the lab runs."""

    def __enter__(self):
        self._stdout = sys.stdout
        sys.stdout = self
        return self

    def __exit__(self, *exc):
        sys.stdout = self._stdout
        return False

    def write(self, text):
        self._stdout.write(text)
        return super().write(text)


console = ConsoleTee()

with console:
    print("======================================================================")
    print("Lab 6: Governed Agent Runtime (OpenCLA / OpenClaw Style)")
    print("======================================================================")

    memory = Memory()
    audit = Audit()
    runtime = GrantRuntime(memory=memory, audit=audit)

    # 1. Successful Run: summarize_literature
    print("=== Run 1: Summarize Literature Skill ===")
    res1 = receive_task(
        {"skill": "summarize_literature", "abstract": "A prospective study on federated learning for eye disease screening."},
        user="suneeta",
        runtime=runtime,
    )
    print(f"Result: {res1}")

    # 2. Successful Run: formulate_budget with tools
    print("\n=== Run 2: Formulate Budget Skill (Calls 2 Tools) ===")
    res2 = receive_task(
        {"skill": "formulate_budget", "direct_cost": 4500000, "agency": "SERB"},
        user="suneeta",
        runtime=runtime,
    )
    print(f"Result: {res2}")

    # 3. Post-Hoc Skill Registration (Zero Changes to Runtime Loop)
    print("\n=== Run 3: Post-Hoc Registered Third Skill ===")
    SKILL_REGISTRY["check_compliance"] = CheckComplianceDocSkill
    res3 = receive_task(
        {"skill": "check_compliance", "doc_type": "Institutional Ethics Clearance"},
        user="suneeta",
        runtime=runtime,
    )
    print(f"Result: {res3}")

    # 4. Assignments
    run_assignment_1_rate_limiting(runtime)
    run_assignment_2_two_failure_escalation(runtime)

    # 5. Inspection of Scoped Memory and Checkpoints
    print("\n=== Scoped Memory for User 'suneeta' ===")
    print(memory.scope("suneeta"))

    print("\n=== Runtime Checkpoints Captured ===")
    for ckpt in runtime.checkpoints:
        print(f"  {ckpt['id']} | {ckpt['label']} | Snapshot: {ckpt['snapshot']}")

Lab 6: Governed Agent Runtime (OpenCLA / OpenClaw Style)
=== Run 1: Summarize Literature Skill ===
Result: Literature summary: Abstract has 10 words. Key finding: Federated screening preserves patient privacy.

=== Run 2: Formulate Budget Skill (Calls 2 Tools) ===
Result: {'direct': 4500000, 'overhead': 675000, 'total': 5175000, 'ceiling': 7500000}

=== Run 3: Post-Hoc Registered Third Skill ===
Result: Compliance check: Verified Institutional Ethics Clearance against institutional ethics policy.

--- [Assignment 1] Per-User Session Rate Limiting ---
Completed 3 valid tasks for prof_sharma.
PASS: 4th task rejected as expected: 'User 'prof_sharma' session rate limit exceeded (maximum 3 tasks per session)'

--- [Assignment 2] Escalate to Human After Two Failures ---
Two consecutive failures recorded in audit log.
Attempt 3 Result: [ESCALATED TO HUMAN AUTHORITY]: Skill 'formulate_budget' failed 2 times previously for user 'researcher_bob'.

=== Scoped Memory for User 'suneeta' ===
{'last_

### Cell 16 · Acceptance checks and deliverables

Each check asserts on the trace of the run above. The trace and the check report are exported to `lab6_artifacts/` (committed under `Course-Project/evidence/`).

In [16]:
import json
import re
from pathlib import Path

LAB = 6
CHECKS = [('Budget skill runs two tools inside the runtime', "'total': 5175000", 1),
    ('A third skill is registered after the runtime is built', 'Post-Hoc Registered Third Skill', 1),
    ('Per-user session rate limit rejects the 4th task', 'PASS: 4th task rejected as expected', 1),
    ('Two failures escalate to the human authority', 'ESCALATED TO HUMAN AUTHORITY', 1),
    ('Memory is scoped per user', 'Scoped Memory for User', 1),
    ('Checkpoints are captured after each skill', 'CKPT-\\d{3} \\| after_', 3)]

trace = console.getvalue()
universal = [
    ("Run produced console output", r"\S", 1),
    ("Graded Assignment 1 executed", r"\[Assignment 1\]", 1),
    ("Graded Assignment 2 executed", r"\[Assignment 2\]", 1),
]
results = []
for label, pattern, minimum in universal + CHECKS:
    found = len(re.findall(pattern, trace, flags=re.MULTILINE))
    results.append({"check": label, "pattern": pattern, "expected_min": minimum,
                    "found": found, "passed": found >= minimum})
no_errors = "Traceback (most recent call last)" not in trace
results.append({"check": "No unhandled exception in the run", "pattern": "Traceback",
                "expected_min": 0, "found": 0 if no_errors else 1, "passed": no_errors})

for i, r in enumerate(results, 1):
    print(f"[{'PASS' if r['passed'] else 'FAIL'}] {i:>2}. {r['check']}")
passed = sum(r["passed"] for r in results)
print(f"\nLab {LAB}: {passed}/{len(results)} acceptance checks passed")

outdir = Path(f"lab{LAB}_artifacts")
outdir.mkdir(exist_ok=True)
(outdir / f"lab{LAB}_console_trace.txt").write_text(trace, encoding="utf-8")
(outdir / f"lab{LAB}_acceptance_checks.json").write_text(
    json.dumps({"lab": LAB, "passed": passed, "total": len(results), "checks": results}, indent=2),
    encoding="utf-8")
print("Artifacts written to", outdir.as_posix())

assert passed == len(results), "acceptance checks failed"

[PASS]  1. Run produced console output
[PASS]  2. Graded Assignment 1 executed
[PASS]  3. Graded Assignment 2 executed
[PASS]  4. Budget skill runs two tools inside the runtime
[PASS]  5. A third skill is registered after the runtime is built
[PASS]  6. Per-user session rate limit rejects the 4th task
[PASS]  7. Two failures escalate to the human authority
[PASS]  8. Memory is scoped per user
[PASS]  9. Checkpoints are captured after each skill
[PASS] 10. No unhandled exception in the run

Lab 6: 10/10 acceptance checks passed
Artifacts written to lab6_artifacts
